# Klasifikasi Produktivitas Padi di Sumatera Berdasarkan Data Iklim

**Proyek Machine Learning Fundamental — Supervised Learning (CPMK-3)**  
S1 Ilmu Komputer • Fasilkom-TI USU • Semester Ganjil TA 2026/2027

## Gambaran Proyek

Proyek ini mempelajari apakah data iklim, provinsi, dan tahun dapat digunakan untuk mengklasifikasikan produktivitas padi di Pulau Sumatera ke dalam dua kategori: **Tinggi** dan **Rendah**. Dataset mencakup delapan provinsi dan catatan tahunan pada rentang 1993–2020, dengan informasi produksi serta luas panen dari BPS dan data iklim dari BMKG.

Produktivitas dihitung dari produksi padi dibagi luas panen, lalu dibentuk menjadi label klasifikasi berdasarkan median produktivitas dataset. Fitur yang digunakan untuk prediksi adalah provinsi, tahun, curah hujan, kelembapan, dan suhu rata-rata. Produksi serta luas panen tidak digunakan sebagai fitur karena keduanya dipakai untuk membentuk label.

Empat algoritma dibandingkan: K-Nearest Neighbors (KNN), Gaussian Naive Bayes, Decision Tree, dan Support Vector Machine (SVM). Evaluasi dilakukan menggunakan Stratified 5-Fold Cross-Validation serta pengujian akhir pada test set. Selain mengukur performa, notebook juga menganalisis fitur yang berpengaruh, pola kesalahan, dan perbedaan performa antarprovinsi.

## Alur Proyek

1. Data Collection
2. Preprocessing
3. Pembagian data train, validation, dan test
4. Training dan pemilihan hyperparameter
5. Evaluasi dan analisis hasil
6. Deployment serta monitoring model

## Sumber Data dan Etika Penggunaan

Dataset berisi data tahunan tanaman padi di provinsi-provinsi Pulau Sumatera. Informasi produksi dan luas panen berasal dari Badan Pusat Statistik (BPS), sedangkan informasi curah hujan, kelembapan, dan suhu berasal dari Badan Meteorologi, Klimatologi, dan Geofisika (BMKG). Dataset dihimpun dan dipublikasikan melalui Kaggle.

Setiap baris mewakili satu provinsi pada satu tahun, bukan data individual petani. Karena itu, hasil model hanya menggambarkan pola pada tingkat provinsi dan tidak dapat langsung digunakan untuk menyimpulkan kondisi setiap petani atau kabupaten.

Data ini bersifat observasional. Model dapat menemukan hubungan statistik antara kondisi iklim dan kelas produktivitas, tetapi tidak dapat membuktikan bahwa kondisi iklim tertentu secara langsung menyebabkan produktivitas tinggi atau rendah. Faktor lain seperti varietas padi, pupuk, irigasi, dan teknik pertanian tidak seluruhnya tercakup dalam dataset.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RS = 42  # random state, supaya hasil bisa diulang

ModuleNotFoundError: No module named 'numpy'

## Tahap 1 — Data Collection

Proyek menggunakan dataset tanaman padi Sumatera yang berisi catatan tahun 1993–2020. Data mencakup delapan provinsi dan menggabungkan informasi produksi pertanian dengan kondisi iklim.

| Kolom | Penjelasan |
|---|---|
| `Provinsi` | Nama provinsi di Pulau Sumatera tempat data dicatat. |
| `Tahun` | Tahun pencatatan data, dari 1993 sampai 2020. |
| `Produksi` | Total produksi padi dalam ton, berdasarkan data BPS. |
| `Luas Panen` | Luas lahan padi yang dipanen dalam hektare, berdasarkan data BPS. |
| `Curah hujan` | Catatan curah hujan yang berkaitan dengan kondisi pertanian, berdasarkan data BMKG. |
| `Kelembapan` | Kelembapan udara dalam persen, berdasarkan data BMKG. |
| `Suhu rata-rata` | Suhu udara rata-rata dalam derajat Celsius, berdasarkan data BMKG. |

Notebook membaca salinan CSV dari alamat internet menggunakan Python. Dengan cara ini, data dimuat langsung saat notebook dijalankan tanpa perlu memilih file secara manual. Jika sumber utama tidak dapat diakses, notebook menyediakan mekanisme cadangan sesuai kode pemuatan data.

In [ ]:
# Sumber data langsung dari internet (tanpa download manual)
URL_DATA = [
    "https://raw.githubusercontent.com/AlrezaAmror/Produksi-Padi/main/Data_Tanaman_Padi_Sumatera_version_1.csv",
    "https://github.com/AlrezaAmror/Produksi-Padi/raw/refs/heads/main/Data_Tanaman_Padi_Sumatera_version_1.csv",
]

path_csv = None
for url in URL_DATA:
    try:
        raw = pd.read_csv(url)
        path_csv = url
        break
    except Exception as e:
        print("Gagal baca", url, "->", type(e).__name__)

if path_csv is None:   # cadangan: file lokal / upload (hanya jika internet/GitHub bermasalah)
    lokal = [p for p in list(Path(".").glob("*.csv")) + list(Path("/content").glob("*.csv"))
             if "padi" in p.name.lower()]
    if not lokal:
        try:
            from google.colab import files
            files.upload()
            lokal = [p for p in Path(".").glob("*.csv") if "padi" in p.name.lower()]
        except ImportError:
            pass
    if not lokal:
        raise FileNotFoundError("Data tidak bisa diambil. Cek koneksi internet atau unduh CSV dari link Kaggle.")
    path_csv = lokal[0]
    raw = pd.read_csv(path_csv)

# Samakan nama kolom: huruf kecil, spasi/strip -> underscore
raw.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in raw.columns]
print("Sumber:", path_csv)
print("Ukuran:", raw.shape)
print("Kolom :", list(raw.columns))
raw.head()

### Membuat Label (Target)

Dataset awal mencatat produksi padi dalam ton, sehingga target penelitian dibentuk menjadi masalah klasifikasi. Pertama, produktivitas dihitung dengan membagi total produksi dengan luas panen:

**Produktivitas = Produksi ÷ Luas Panen (ton/ha)**

Nilai produktivitas kemudian dibagi menjadi dua kelas berdasarkan median produktivitas seluruh data:
- **Tinggi**: produktivitas sama dengan atau lebih besar daripada median.
- **Rendah**: produktivitas lebih kecil daripada median.

Pembagian berdasarkan median membuat dua kelas ditentukan dari distribusi data, bukan dari batas produktivitas yang ditetapkan secara eksternal. Kelas `Tinggi` digunakan sebagai kelas positif dalam evaluasi.

Kolom `Produksi` dan `Luas Panen` tidak digunakan sebagai fitur model karena keduanya dipakai untuk menghitung target. Jika kolom tersebut ikut dimasukkan, model dapat memperoleh jawaban secara tidak langsung dan evaluasinya menjadi tidak adil. Fitur prediksi yang digunakan adalah provinsi, tahun, curah hujan, kelembapan, dan suhu rata-rata.

In [ ]:
perlu = ["provinsi", "tahun", "produksi", "luas_panen", "curah_hujan", "kelembapan", "suhu_rata_rata"]
hilang = [c for c in perlu if c not in raw.columns]
assert not hilang, f"Kolom tidak ditemukan: {hilang}. Cek nama kolom CSV."

df = raw[perlu].copy()
df["produktivitas"] = df["produksi"] / df["luas_panen"]          # ton per hektar
print("Provinsi:", sorted(df["provinsi"].unique()))
print("Tahun   :", df["tahun"].min(), "-", df["tahun"].max())
df.describe().T.round(2)

### Eksplorasi Data (EDA)

In [ ]:
print("Nilai kosong per kolom:\n", df.isna().sum(), "\n")
print("Jumlah baris per provinsi:\n", df["provinsi"].value_counts())

plt.figure(figsize=(10, 4.5))
sns.lineplot(data=df, x="tahun", y="produktivitas", hue="provinsi", marker="o", markersize=3)
plt.ylabel("Produktivitas (ton/ha)"); plt.title("Produktivitas padi per provinsi, 1993–2020")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()

## Tahap 2 — Preprocessing

Preprocessing menyiapkan data agar dapat digunakan oleh algoritma machine learning.

1. **Pembersihan data:** baris duplikat dihapus. Nilai yang berada di luar rentang masuk akal berdasarkan pemeriksaan domain ditandai sebagai nilai kosong agar tidak memengaruhi model.
2. **Penanganan nilai kosong:** nilai numerik yang kosong diisi dengan median kolom terkait. Median relatif tahan terhadap nilai ekstrem dibandingkan rata-rata.
3. **Scaling:** fitur numerik distandardisasi menggunakan `StandardScaler`. Langkah ini penting untuk KNN dan SVM karena keduanya sensitif terhadap skala fitur; misalnya curah hujan dapat memiliki rentang jauh lebih besar daripada suhu.
4. **Encoding:** nama provinsi diubah menjadi representasi numerik menggunakan One-Hot Encoding agar dapat diproses oleh algoritma.

Langkah imputasi, scaling, dan encoding ditempatkan dalam `Pipeline`. Dengan demikian, parameter preprocessing dipelajari dari data pelatihan pada setiap proses evaluasi dan tidak menggunakan informasi dari data uji.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

# Aturan domain (batas longgar, hanya untuk menangkap salah catat)
batas = {"tahun": (1990, 2030), "curah_hujan": (0, 10000), "kelembapan": (0, 100),
         "suhu_rata_rata": (15, 35), "produktivitas": (0.5, 12)}
for kolom, (lo, hi) in batas.items():
    salah = df[kolom].notna() & ~df[kolom].between(lo, hi)
    print(f"{kolom:16s} di luar batas: {salah.sum()}")
    df.loc[salah, kolom] = np.nan

df = df.dropna(subset=["produktivitas"]).reset_index(drop=True)   # label harus ada
print("\nNilai kosong pada fitur:\n", df[["tahun", "curah_hujan", "kelembapan", "suhu_rata_rata"]].isna().sum())

# Label: Tinggi (1) jika >= median
MEDIAN = df["produktivitas"].median()
df["kelas"] = np.where(df["produktivitas"] >= MEDIAN, "Tinggi", "Rendah")
print(f"\nMedian produktivitas = {MEDIAN:.2f} ton/ha")
print(df["kelas"].value_counts())

fitur = ["provinsi", "tahun", "curah_hujan", "kelembapan", "suhu_rata_rata"]
X = df[fitur]
y = (df["kelas"] == "Tinggi").astype(int)      # 1 = Tinggi (positif), 0 = Rendah
kol_num = [c for c in fitur if c != "provinsi"]
kol_cat = ["provinsi"]

def buat_preprocessor(scaling=True):
    langkah = [("imp", SimpleImputer(strategy="median"))]
    if scaling:
        langkah.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(langkah), kol_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), kol_cat),
    ])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, f in zip(axes, ["curah_hujan", "kelembapan", "suhu_rata_rata"]):
    sns.boxplot(data=df, x="kelas", y=f, order=["Rendah", "Tinggi"], palette=["#e9c46a", "#2a9d8f"], ax=ax)
    ax.set_title(f)
plt.suptitle("Sebaran fitur iklim per kelas produktivitas", y=1.03)
plt.tight_layout(); plt.show()

## Tahap 3 — Pembagian Data 70/15/15

Dataset dibagi menjadi tiga bagian: sekitar 70% untuk pelatihan, 15% untuk validasi, dan 15% untuk pengujian akhir. Pembagian menggunakan stratifikasi agar proporsi kelas `Tinggi` dan `Rendah` relatif terjaga di setiap bagian.

- **Data train** digunakan untuk melatih model.
- **Data validation** digunakan untuk memilih hyperparameter yang sesuai tanpa memakai data test.
- **Data test** disimpan untuk evaluasi akhir setelah pemilihan model selesai.

Pemisahan ini membantu mengukur kemampuan model pada data yang tidak digunakan saat pemilihan hyperparameter. Namun, karena dataset merupakan data deret waktu dan pembagiannya dilakukan secara acak, evaluasi ini memiliki keterbatasan: catatan dari tahun yang berdekatan dapat masuk ke bagian train dan test yang berbeda.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=RS)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=RS)

for nama, yy in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{nama:5s}: {len(yy):3d} data | % Tinggi = {yy.mean()*100:.1f}")

## Tahap 4 — Training

Empat algoritma klasifikasi dibandingkan untuk mengetahui pendekatan mana yang paling baik membedakan produktivitas `Tinggi` dan `Rendah`. Pemilihan hyperparameter dilakukan berdasarkan nilai F1 pada data validation.

| Algoritma | Pengaturan yang diuji |
|---|---|
| K-Nearest Neighbors (KNN) | Jumlah tetangga `K` ganjil dari 1 sampai 25. |
| Gaussian Naive Bayes | Menggunakan asumsi distribusi Gaussian untuk fitur numerik. |
| Decision Tree | Kedalaman pohon dan jumlah minimum sampel pada daun dibatasi untuk mengurangi overfitting. |
| Support Vector Machine (SVM) | Kernel RBF dengan pengujian nilai `C` dan `gamma`. |

KNN mengklasifikasikan data berdasarkan tetangga terdekat. Naive Bayes menggunakan probabilitas berdasarkan fitur. Decision Tree membentuk aturan keputusan bertingkat, sedangkan SVM mencari batas pemisah antarkelas. Keempat model memiliki cara belajar yang berbeda sehingga perbandingan dapat menunjukkan pendekatan yang paling sesuai dengan dataset ini.

In [ ]:
def pipa(clf, scaling=True):
    return Pipeline([("pre", buat_preprocessor(scaling)), ("clf", clf)])

def f1_val(clf, scaling=True):
    p = pipa(clf, scaling).fit(X_train, y_train)
    return f1_score(y_val, p.predict(X_val))

# --- KNN: Elbow method (K ganjil) ---
daftar_k = list(range(1, 26, 2))
f1_k = [f1_val(KNeighborsClassifier(n_neighbors=k)) for k in daftar_k]
K_TERBAIK = daftar_k[int(np.argmax(f1_k))]

plt.figure(figsize=(6, 3.5))
plt.plot(daftar_k, [1 - s for s in f1_k], marker="o", color="#2a9d8f")
plt.axvline(K_TERBAIK, ls="--", color="gray"); plt.xticks(daftar_k)
plt.xlabel("K"); plt.ylabel("Error (1 - F1) pada validation"); plt.title("Elbow Method KNN")
plt.tight_layout(); plt.show()

# --- Decision Tree: pruning (tanpa scaling, pohon tidak membutuhkannya) ---
best, DT_PARAM = -1, None
for d in [3, 4, 5, 6]:
    for leaf in [1, 3, 5, 10]:
        s = f1_val(DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, random_state=RS), scaling=False)
        if s > best:
            best, DT_PARAM = s, dict(max_depth=d, min_samples_leaf=leaf)

# --- SVM RBF ---
best, SVM_PARAM = -1, None
for C in [0.1, 1, 10, 100]:
    for g in ["scale", 0.01, 0.1, 1]:
        s = f1_val(SVC(kernel="rbf", C=C, gamma=g, random_state=RS))
        if s > best:
            best, SVM_PARAM = s, dict(C=C, gamma=g)

print("K terbaik (KNN)    :", K_TERBAIK)
print("Param Decision Tree:", DT_PARAM)
print("Param SVM (RBF)    :", SVM_PARAM)

In [ ]:
model = {
    "KNN": pipa(KNeighborsClassifier(n_neighbors=K_TERBAIK, metric="euclidean")),
    "Naive Bayes": pipa(GaussianNB()),
    "Decision Tree": pipa(DecisionTreeClassifier(random_state=RS, **DT_PARAM), scaling=False),
    "SVM (RBF)": pipa(SVC(kernel="rbf", random_state=RS, **SVM_PARAM)),
}
print("Keempat model siap:", list(model))

## Perbandingan dengan 5-Fold Cross-Validation

Setelah hyperparameter dipilih, data train dan validation digabungkan menjadi data pengembangan yang mencakup sekitar 85% dataset. Data test tetap disimpan dan tidak digunakan pada tahap ini.

Stratified 5-Fold Cross-Validation membagi data pengembangan menjadi lima bagian dengan proporsi kelas yang dijaga. Pada setiap putaran, empat bagian digunakan untuk melatih model dan satu bagian digunakan untuk evaluasi. Proses ini diulang hingga setiap bagian pernah menjadi data evaluasi.

Hasil dari lima putaran diringkas menggunakan rata-rata dan standar deviasi. Accuracy mengukur proporsi prediksi yang benar, sedangkan precision, recall, dan F1-score membantu melihat kualitas prediksi per kelas. F1-macro menghitung F1 setiap kelas dengan bobot yang sama sehingga kelas `Tinggi` dan `Rendah` sama-sama diperhitungkan.

In [ ]:
X_dev = pd.concat([X_train, X_val]); y_dev = pd.concat([y_train, y_val])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
metrik = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}

baris, f1_semua = [], {}
for nama, m in model.items():
    h = cross_validate(m, X_dev, y_dev, cv=cv, scoring=metrik)
    f1_semua[nama] = h["test_f1"]
    baris.append({"Model": nama,
                  **{k.capitalize(): f"{h['test_'+k].mean():.3f} ± {h['test_'+k].std():.3f}" for k in metrik},
                  "_f1": h["test_f1"].mean()})

tabel_cv = pd.DataFrame(baris).sort_values("_f1", ascending=False)
MODEL_TERBAIK = tabel_cv.iloc[0]["Model"]
display(tabel_cv.drop(columns="_f1").reset_index(drop=True))
print("Model terbaik (F1 CV tertinggi):", MODEL_TERBAIK)

plt.figure(figsize=(7, 3.8))
sns.boxplot(data=pd.DataFrame(f1_semua), palette="Greens")
sns.stripplot(data=pd.DataFrame(f1_semua), color="black", size=4)
plt.ylabel("F1 per fold"); plt.title("F1-score 5-Fold CV")
plt.tight_layout(); plt.show()

## Tahap 5 — Evaluation pada Test Set

Setelah model dan hyperparameter dipilih, model dilatih ulang menggunakan gabungan data train dan validation. Model kemudian dievaluasi satu kali pada data test yang belum digunakan selama pemilihan model.

Evaluasi menampilkan accuracy, precision, recall, F1-score, classification report, dan confusion matrix. Confusion matrix memperlihatkan jumlah prediksi benar dan salah untuk kelas `Rendah` serta `Tinggi`.

Dalam penelitian ini, kelas `Tinggi` diperlakukan sebagai kelas positif. False positive berarti model memprediksi produktivitas tinggi ketika label sebenarnya rendah. False negative berarti model memprediksi produktivitas rendah ketika label sebenarnya tinggi. Kedua jenis kesalahan memiliki makna berbeda sehingga perlu dilihat bersama metrik agregat.

In [ ]:
hasil, prediksi = [], {}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (nama, m) in zip(axes, model.items()):
    m.fit(X_dev, y_dev)
    p = m.predict(X_test); prediksi[nama] = p
    hasil.append({"Model": nama,
                  "Accuracy": accuracy_score(y_test, p), "Precision": precision_score(y_test, p),
                  "Recall": recall_score(y_test, p), "F1": f1_score(y_test, p)})
    ConfusionMatrixDisplay(confusion_matrix(y_test, p), display_labels=["Rendah", "Tinggi"]).plot(
        ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(nama)
plt.tight_layout(); plt.show()

tabel_test = pd.DataFrame(hasil).set_index("Model").round(3).sort_values("F1", ascending=False)
tabel_test

In [ ]:
print(f"Laporan rinci — {MODEL_TERBAIK}")
print(classification_report(y_test, prediksi[MODEL_TERBAIK], target_names=["Rendah", "Tinggi"]))

## Analisis dan Pembahasan

Bagian ini membantu memahami bukan hanya skor model, tetapi juga bagaimana model membuat prediksi dan faktor apa yang berkaitan dengan hasilnya.

In [ ]:
dt = model["Decision Tree"]
nama_fitur = [n.split("__")[1] for n in dt.named_steps["pre"].get_feature_names_out()]

plt.figure(figsize=(15, 6))
plot_tree(dt.named_steps["clf"], feature_names=nama_fitur, class_names=["Rendah", "Tinggi"],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title("Decision Tree (sampai kedalaman 3; satuan asli, tanpa scaling)")
plt.show()

print(export_text(dt.named_steps["clf"], feature_names=nama_fitur, max_depth=3))

### 2. Fitur yang Paling Berpengaruh

Permutation importance digunakan untuk memperkirakan seberapa besar kontribusi setiap fitur terhadap performa model terbaik. Caranya, nilai pada satu fitur diacak, lalu perubahan skor F1 diukur. Jika F1 menurun cukup besar setelah fitur diacak, model kemungkinan banyak memanfaatkan informasi dari fitur tersebut.

Nilai ini menunjukkan kepentingan fitur bagi prediksi model, bukan bukti bahwa fitur tersebut menyebabkan produktivitas berubah. Hasilnya juga dapat dipengaruhi oleh hubungan antarfitur dan ukuran dataset.

In [ ]:
pi = permutation_importance(model[MODEL_TERBAIK], X_test, y_test, scoring="f1",
                            n_repeats=30, random_state=RS)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
plt.figure(figsize=(6, 3.5))
imp.plot.barh(color="#2a9d8f", xerr=pd.Series(pi.importances_std, index=X_test.columns)[imp.index])
plt.xlabel("Penurunan F1 jika fitur diacak"); plt.title(f"Pentingnya fitur — {MODEL_TERBAIK}")
plt.tight_layout(); plt.show()

### 3. Cara Membaca Hasil Perbandingan

Beberapa hal yang dapat diamati dari tabel, grafik, dan evaluasi model:

- **Scaling fitur:** KNN dan SVM sensitif terhadap perbedaan skala karena menggunakan jarak atau margin, sedangkan Decision Tree memilih pemisahan berdasarkan ambang fitur.
- **Accuracy dan F1:** model dengan accuracy tertinggi belum tentu memiliki F1-macro tertinggi, terutama jika performa pada dua kelas berbeda.
- **Pengaruh fitur:** permutation importance membantu membandingkan peran kondisi iklim, provinsi, dan tahun terhadap prediksi model.
- **Jenis kesalahan:** false positive dan false negative memiliki konsekuensi interpretasi berbeda ketika model digunakan untuk mengenali wilayah dengan produktivitas rendah.
- **Kestabilan hasil:** standar deviasi skor antar-fold menunjukkan seberapa banyak performa model berubah pada pembagian data yang berbeda. Perbedaan rata-rata yang kecil perlu ditafsirkan dengan hati-hati.

## Etika AI — Pemeriksaan Performa per Provinsi

Model yang terlihat baik secara keseluruhan belum tentu memiliki performa yang sama untuk setiap provinsi. Karena itu, hasil prediksi cross-validation diringkas per provinsi untuk melihat apakah kesalahan model terkonsentrasi pada wilayah tertentu.

Pemeriksaan ini bukan penilaian bahwa suatu provinsi lebih baik atau lebih buruk. Tujuannya adalah menemukan kemungkinan ketimpangan performa akibat jumlah data yang terbatas atau pola data yang berbeda antarwilayah.

In [ ]:
pred_cv = cross_val_predict(model[MODEL_TERBAIK], X, y, cv=cv)
cek = X[["provinsi"]].assign(y=y.values, p=pred_cv)

def ringkas(g):
    return pd.Series({
        "Jumlah": len(g),
        "% Tinggi asli": g.y.mean() * 100,
        "Accuracy": accuracy_score(g.y, g.p),
        "Recall Tinggi": recall_score(g.y, g.p, zero_division=0),
        "Precision Tinggi": precision_score(g.y, g.p, zero_division=0),
    })

adil = cek.groupby("provinsi").apply(ringkas).round(2)
display(adil)

adil["Accuracy"].sort_values().plot.barh(color="#2a9d8f", figsize=(6, 3.8))
plt.axvline(accuracy_score(y, pred_cv), ls="--", color="gray", label="rata-rata semua provinsi")
plt.xlabel("Accuracy"); plt.title("Akurasi model per provinsi"); plt.legend()
plt.tight_layout(); plt.show()

### Keterbatasan dan Pertimbangan Etika

- Setiap provinsi hanya memiliki sekitar 28 catatan tahunan, sehingga metrik per provinsi dapat berubah cukup besar dan perlu ditafsirkan sebagai indikasi, bukan kesimpulan pasti.
- Dataset memiliki struktur deret waktu, tetapi pembagian acak dapat menempatkan tahun yang berdekatan pada train dan test. Karena itu, skor bisa lebih optimistis daripada pengujian untuk memprediksi tahun mendatang. Pengembangan berikutnya dapat melatih model pada tahun sampai 2015 dan menguji pada tahun setelah 2015.
- Data merepresentasikan provinsi, bukan petani individual atau setiap kabupaten. Prediksi tidak tepat digunakan untuk menilai individu atau lokasi yang lebih rinci daripada cakupan datanya.
- Model hanya alat bantu analisis. Keputusan pertanian dan kebijakan pangan perlu mempertimbangkan pengetahuan ahli serta faktor lain yang tidak tercatat di dataset.
- Hubungan statistik bukan bukti sebab-akibat. Produktivitas padi juga dapat dipengaruhi oleh pupuk, varietas, irigasi, hama, teknologi, dan praktik budidaya yang tidak tercakup dalam fitur.

## Tahap 6 — Deployment dan Monitoring

Pada tahap deployment, model terbaik dilatih ulang menggunakan seluruh data yang tersedia dan disimpan ke berkas `model_produktivitas_padi.joblib`. Notebook juga menyediakan fungsi prediksi sederhana yang menerima informasi provinsi dan kondisi iklim, lalu mengembalikan perkiraan kelas produktivitas `Tinggi` atau `Rendah`.

Prediksi merupakan hasil estimasi berdasarkan pola yang dipelajari dari data historis, bukan jaminan produktivitas aktual. Penggunaan pada kondisi atau tahun yang berbeda dari data pelatihan perlu dilakukan secara hati-hati.

In [ ]:
final_model = model[MODEL_TERBAIK].fit(X, y)
joblib.dump(final_model, "model_produktivitas_padi.joblib")
print("Tersimpan: model_produktivitas_padi.joblib ->", MODEL_TERBAIK)

def prediksi_produktivitas(provinsi, tahun, curah_hujan, kelembapan, suhu_rata_rata):
    m = joblib.load("model_produktivitas_padi.joblib")
    x = pd.DataFrame([dict(provinsi=provinsi, tahun=tahun, curah_hujan=curah_hujan,
                           kelembapan=kelembapan, suhu_rata_rata=suhu_rata_rata)])
    hasil = "Tinggi 🌾" if m.predict(x)[0] == 1 else "Rendah ⚠️"
    if hasattr(m, "predict_proba"):
        hasil += f" (keyakinan {m.predict_proba(x)[0].max()*100:.0f}%)"
    return hasil

# Demo: ambil satu baris Sumut dari data sebagai contoh input
sumut = df[df["provinsi"].str.contains("Utara", case=False)].iloc[-1]
print("Provinsi :", sumut["provinsi"], "| Tahun:", int(sumut["tahun"]))
print("Prediksi :", prediksi_produktivitas(sumut["provinsi"], sumut["tahun"], sumut["curah_hujan"],
                                          sumut["kelembapan"], sumut["suhu_rata_rata"]))
print("Aktual   :", sumut["kelas"], f"({sumut['produktivitas']:.2f} ton/ha)")

### Monitoring Model

Performa model perlu diperiksa kembali ketika data tahunan terbaru tersedia. Data produksi dan luas panen dari BPS serta informasi iklim dari BMKG dapat ditambahkan setelah melalui proses pembersihan dan penyelarasan format. Prediksi kemudian dibandingkan dengan hasil aktual untuk memantau perubahan accuracy, precision, recall, dan F1-score.

Jika performa menurun atau pola data berubah, model dapat dilatih ulang. Monitoring juga perlu memeriksa performa per provinsi agar penurunan pada wilayah tertentu tidak tertutup oleh skor rata-rata keseluruhan.

## Kesimpulan

Sel ini merangkum jumlah data yang dianalisis, nilai median produktivitas, serta model yang memperoleh hasil terbaik pada evaluasi. Kesimpulan perlu dibaca bersama tabel metrik dan confusion matrix karena satu skor saja tidak menggambarkan seluruh perilaku model.

Model dengan F1 terbaik adalah model yang paling seimbang dalam mengenali kelas `Tinggi` dan `Rendah` pada evaluasi yang dilakukan. Hasil tersebut hanya berlaku untuk dataset dan skenario pengujian ini, sehingga tidak otomatis menjamin performa yang sama pada data dari tahun mendatang.

In [ ]:
terbaik_test = tabel_test.index[0]
print(f"Dataset              : Tanaman Padi Sumatera (Kaggle; BPS + BMKG), n = {len(df)}")
print(f"Median produktivitas : {MEDIAN:.2f} ton/ha (batas Tinggi/Rendah)")
print(f"Model terbaik (CV F1): {MODEL_TERBAIK}")
print(f"Model terbaik (test) : {terbaik_test} | F1 = {tabel_test.loc[terbaik_test, 'F1']:.3f}")
print("\nRingkasan F1 5-Fold CV:")
for n, s in sorted(f1_semua.items(), key=lambda kv: -kv[1].mean()):
    print(f"  {n:14s} {s.mean():.3f} ± {s.std():.3f}")

## Ringkasan Alur Proyek

Notebook ini menjalankan enam tahap utama machine learning:

1. **Data Collection:** memuat data produksi padi, luas panen, dan kondisi iklim untuk provinsi di Sumatera.
2. **Preprocessing:** membersihkan data, menangani nilai kosong, menstandarkan fitur numerik, dan mengubah provinsi menjadi fitur numerik.
3. **Split Data:** membagi data menjadi train, validation, dan test agar pemilihan model terpisah dari evaluasi akhir.
4. **Training:** melatih dan memilih hyperparameter untuk KNN, Naive Bayes, Decision Tree, dan SVM.
5. **Evaluation:** membandingkan model menggunakan 5-fold cross-validation serta menguji model terpilih pada test set.
6. **Deployment & Monitoring:** menyimpan model terbaik, menyediakan fungsi prediksi, dan menjelaskan cara memantau performanya saat data baru tersedia.

Analisis tambahan mencakup struktur pohon keputusan, permutation importance, jenis kesalahan klasifikasi, dan perbandingan performa antarprovinsi.